#  Mini-Transformer (SmallGPT): 

**Generative AI · Model Making **

> **Runtime:** Colab -> Runtime -> Change runtime type -> **T4 GPU**. Full run ~25-35 min. Set `FAST_MODE = True` in the next cell for a ~3-minute smoke test (numbers will be poor, use it only to check the notebook runs).

In [ ]:
import math, time, re, urllib.request, subprocess, sys
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F

FAST_MODE = False          # True -> ~10x fewer iterations (smoke test only)
SEED = 1337

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = (device == "cuda")
print("PyTorch:", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device == "cuda" else "CPU (will be slow)")

ITERS = lambda n: max(200, n // 10) if FAST_MODE else n
torch.manual_seed(SEED)

## 1. Data and character-level tokenizer 

Tiny Shakespeare (~1.1M characters, 65-character vocabulary). We keep the same 90/10 train/validation split so results stay comparable with the first stage.
We also keep a set of *words that really occur in the corpus* — used later as a simple automatic quality metric for generated text.

In [ ]:
data_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
data_path = Path("input.txt")
if not data_path.exists():
    urllib.request.urlretrieve(data_url, data_path)
text = data_path.read_text(encoding="utf-8")

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda ids: "".join(itos[i] for i in ids)

ids_all = torch.tensor(encode(text), dtype=torch.long)
n_split = int(0.9 * len(ids_all))
CHAR_DATA = (ids_all[:n_split], ids_all[n_split:])
train_text, val_text = text[:n_split], text[n_split:]

corpus_words = set(re.findall(r"[a-z']+", text.lower()))
print(f"chars: {len(text):,} | vocab: {vocab_size} | train: {len(CHAR_DATA[0]):,} | val: {len(CHAR_DATA[1]):,}")
print(f"unique words in corpus: {len(corpus_words):,}")

## 2. Refactored, configurable Transformer

Architecture (decoder-only, pre-LayerNorm, causal multi-head attention, GELU MLP, residual connections):

* everything is controlled by one `Config`, so we can run **scaling and ablation experiments** by changing a number;
* attention is computed for all heads at once (faster) using PyTorch's fused causal attention;
* a `store_attn` switch lets us **extract the real attention weights** for visualisation;
* `use_pos_emb=False` lets us **ablate positional embeddings**;
* `generate()` supports **temperature, top-k and top-p**.

The attention formula: `softmax(QKᵀ/√d_k + causal_mask) · V`.

In [ ]:
@dataclass
class Config:
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    block_size: int = 128
    dropout: float = 0.1
    use_pos_emb: bool = True
    vocab_size: int = 65

def make_cfg(**kw):
    return Config(**{"vocab_size": vocab_size, **kw})


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0, "n_embd must be divisible by n_head"
        self.n_head = cfg.n_head
        self.dropout_p = cfg.dropout
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.register_buffer(
            "mask", torch.tril(torch.ones(cfg.block_size, cfg.block_size)).view(1, 1, cfg.block_size, cfg.block_size)
        )
        self.store_attn = False
        self.last_attn = None

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        hs = C // self.n_head
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)   # (B, H, T, hs)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)

        if self.store_attn:   # explicit path so we can keep the attention matrix
            att = (q @ k.transpose(-2, -1)) * (hs ** -0.5)
            att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
            att = F.softmax(att, dim=-1)
            self.last_attn = att.detach().float().cpu()
            y = self.attn_drop(att) @ v
        else:                 # fast fused path
            y = F.scaled_dot_product_attention(
                q, k, v, dropout_p=self.dropout_p if self.training else 0.0, is_causal=True
            )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd), nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd), nn.Dropout(cfg.dropout),
        )
    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(cfg.n_embd), CausalSelfAttention(cfg)
        self.ln2, self.mlp = nn.LayerNorm(cfg.n_embd), MLP(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # LayerNorm -> attention -> residual
        x = x + self.mlp(self.ln2(x))    # LayerNorm -> MLP -> residual
        return x


class MiniGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd) if cfg.use_pos_emb else None
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.pos_emb is not None:
            x = x + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            logits = logits[:, -1, :]
            if temperature == 0:                                   # greedy
                nxt = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                logits = logits / temperature                      # reshape the distribution
                if top_k is not None:                              # keep k most likely
                    kth = torch.topk(logits, min(top_k, logits.size(-1)))[0][:, [-1]]
                    logits = logits.masked_fill(logits < kth, float("-inf"))
                if top_p is not None:                              # nucleus: smallest set with mass >= p
                    s_logits, s_idx = torch.sort(logits, descending=True)
                    probs = F.softmax(s_logits, dim=-1)
                    remove = (torch.cumsum(probs, dim=-1) - probs) > top_p
                    s_logits = s_logits.masked_fill(remove, float("-inf"))
                    logits = torch.full_like(logits, float("-inf")).scatter(1, s_idx, s_logits)
                nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


m = MiniGPT(make_cfg())
print("Baseline parameter count:", f"{sum(p.numel() for p in m.parameters()):,}", "(first-stage model had 824,897)")
del m

## 3. Better training loop

* **Learning-rate warmup + cosine decay** — large steps early are unstable; decaying the rate lets the model settle into a lower loss;
* **Gradient clipping** (norm 1.0) — protects against rare exploding gradients;
* **Mixed precision (fp16 autocast)** on GPU — much faster on a T4;
* **Best-checkpoint saving** by validation loss;
* **Sample snapshots** during training, so we can *show* text quality improving;
* automatic computation of **val loss, perplexity (exp of loss), bits-per-character, train/val gap**.

In [ ]:
def get_batch(split, cfg, batch_size, data):
    src = data[0] if split == "train" else data[1]
    ix = torch.randint(len(src) - cfg.block_size - 1, (batch_size,))
    x = torch.stack([src[i:i + cfg.block_size] for i in ix])
    y = torch.stack([src[i + 1:i + cfg.block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, cfg, batch_size, eval_iters, data):
    model.eval(); out = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, cfg, batch_size, data)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def lr_at(step, max_iters, lr, warmup=100, min_ratio=0.1):
    if step < warmup:
        return lr * (step + 1) / warmup
    prog = (step - warmup) / max(1, max_iters - warmup)
    return lr * (min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * prog)))

def train_model(name, cfg, max_iters=3000, batch_size=64, lr=3e-4, eval_interval=300, eval_iters=50,
                sample_steps=(), data=None, save_best=False, verbose=True):
    data = data if data is not None else CHAR_DATA
    model = MiniGPT(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    hist = {"step": [], "train": [], "val": []}
    snaps, best_val, t0 = {}, float("inf"), time.time()
    if verbose:
        print(f"\n=== {name} | params {n_params:,} | layers {cfg.n_layer} heads {cfg.n_head} "
              f"embd {cfg.n_embd} ctx {cfg.block_size} dropout {cfg.dropout} pos_emb {cfg.use_pos_emb}")

    for step in range(max_iters):
        for g in opt.param_groups:
            g["lr"] = lr_at(step, max_iters, lr)

        if step % eval_interval == 0 or step == max_iters - 1:
            l = estimate_loss(model, cfg, batch_size, eval_iters, data)
            hist["step"].append(step); hist["train"].append(l["train"]); hist["val"].append(l["val"])
            if verbose:
                print(f"step {step:5d} | train {l['train']:.4f} | val {l['val']:.4f}")
            if save_best and l["val"] < best_val:
                best_val = l["val"]
                torch.save({"cfg": asdict(cfg), "state": model.state_dict(), "chars": chars}, f"{name}_best.pt")

        if step in sample_steps:
            ctx = torch.tensor([encode("ROMEO:")], dtype=torch.long, device=device)
            snaps[step] = decode(model.generate(ctx, 200)[0].tolist())
            model.train()

        xb, yb = get_batch("train", cfg, batch_size, data)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()

    minutes = (time.time() - t0) / 60
    tr, va = hist["train"][-1], hist["val"][-1]
    if verbose:
        print(f"done in {minutes:.2f} min | val loss {va:.4f} | perplexity {math.exp(va):.2f} | "
              f"bits/char {va / math.log(2):.3f}")
    return dict(name=name, model=model, cfg=cfg, params=n_params, hist=hist, snaps=snaps, minutes=minutes,
                train_loss=tr, val_loss=va, gap=va - tr, ppl=math.exp(va), bpc=va / math.log(2))

def summarize(res):
    return {"run": res["name"], "params": res["params"], "train_loss": round(res["train_loss"], 4),
            "val_loss": round(res["val_loss"], 4), "gap(val-train)": round(res["gap"], 4),
            "perplexity": round(res["ppl"], 2), "bits/char": round(res["bpc"], 3),
            "minutes": round(res["minutes"], 2)}

def plot_curves(res_list, title):
    plt.figure(figsize=(9, 5))
    for r in res_list:
        plt.plot(r["hist"]["step"], r["hist"]["train"], "--", label=f"{r['name']} train")
        plt.plot(r["hist"]["step"], r["hist"]["val"], "-", label=f"{r['name']} val")
    plt.xlabel("training step"); plt.ylabel("cross-entropy loss"); plt.title(title)
    plt.legend(); plt.grid(alpha=0.25); plt.show()

## 4. Experiment A — Scaling: baseline vs. a bigger model

* **baseline** = the first-stage configuration (4 layers, 4 heads, 128-dim, 128-char context, dropout 0.1).
* **scaled** = 6 layers, 8 heads, 256-dim, 256-char context, dropout 0.2 (more dropout because a bigger model overfits a 1M-character corpus more easily).

Expect the scaled model to reach a lower validation loss — the interesting question is **by how much, at what cost, and whether the train/val gap widens**.

In [ ]:
results = {}

torch.manual_seed(SEED)
N_BASE = ITERS(3000)
results["baseline"] = train_model(
    "baseline", make_cfg(n_layer=4, n_head=4, n_embd=128, block_size=128, dropout=0.1),
    max_iters=N_BASE, eval_interval=max(1, N_BASE // 10), sample_steps=(0, N_BASE // 5, N_BASE - 1))

for step, s in results["baseline"]["snaps"].items():
    print(f"\n----- baseline sample @ step {step} -----\n{s[:200]}")

In [ ]:
torch.manual_seed(SEED)
N_SCALED = ITERS(4000)
results["scaled"] = train_model(
    "scaled", make_cfg(n_layer=6, n_head=8, n_embd=256, block_size=256, dropout=0.2),
    max_iters=N_SCALED, eval_interval=max(1, N_SCALED // 10), sample_steps=(0, N_SCALED // 5, N_SCALED - 1),
    save_best=True)

for step, s in results["scaled"]["snaps"].items():
    print(f"\n----- scaled sample @ step {step} -----\n{s[:200]}")

In [ ]:
plot_curves([results["baseline"], results["scaled"]], "Scaling: baseline vs scaled model")
display(pd.DataFrame([summarize(results["baseline"]), summarize(results["scaled"])]).set_index("run"))

### How to read this (write your own numbers into the report)

* **Perplexity** = e^(loss): "the model is as uncertain as if choosing uniformly among *this many* characters". A random guesser over 65 characters has perplexity 65.
* **Bits/char** = loss / ln 2 — the standard unit for character-level language models; lower is better.
* **Gap (val − train)** — a growing gap means the model is starting to memorise the training text. Compare the baseline gap with the scaled gap.
* Note the scaled model sees a 256-character context vs 128, so the comparison mixes *capacity* and *context*. The ablation study below separates single factors.

## 5. Experiment B — Ablation study: which components matter?

Starting from the baseline architecture, we change **one thing at a time** and train each variant for the same number of steps with the same seed.
This is how you *prove* a component matters rather than just describe it.

| Variant | What changes | Hypothesis to test |
|---|---|---|
| full | nothing | reference |
| no_pos_emb | positional embeddings removed | Self-attention is order-blind, so quality should drop... but **causal masking itself leaks some position information** — see what actually happens |
| 1_head | 1 attention head | Fewer relationship types → worse |
| 8_heads | 8 heads (16-dim each) | Little change (capacity same, split differently) |
| 1_layer / 2_layers | depth | Depth is where composition of patterns happens |

In [ ]:
base_kw = dict(n_layer=4, n_head=4, n_embd=128, block_size=128, dropout=0.1)
variants = {
    "full":       make_cfg(**base_kw),
    "no_pos_emb": make_cfg(**{**base_kw, "use_pos_emb": False}),
    "1_head":     make_cfg(**{**base_kw, "n_head": 1}),
    "8_heads":    make_cfg(**{**base_kw, "n_head": 8}),
    "1_layer":    make_cfg(**{**base_kw, "n_layer": 1}),
    "2_layers":   make_cfg(**{**base_kw, "n_layer": 2}),
}
N_ABL = ITERS(1500)
ablation = {}
for name, cfg in variants.items():
    torch.manual_seed(SEED)                       # identical init/data order for a fair comparison
    ablation[name] = train_model(f"abl_{name}", cfg, max_iters=N_ABL, eval_interval=max(1, N_ABL // 5),
                                 eval_iters=50, verbose=False)
    r = ablation[name]
    print(f"{name:11s} | params {r['params']:>9,} | val loss {r['val_loss']:.4f} | ppl {r['ppl']:.2f} | {r['minutes']:.2f} min")

In [ ]:
abl_df = pd.DataFrame([{**summarize(r), "variant": k} for k, r in ablation.items()]).set_index("variant")
abl_df["delta_vs_full"] = (abl_df["val_loss"] - abl_df.loc["full", "val_loss"]).round(4)
display(abl_df[["params", "train_loss", "val_loss", "delta_vs_full", "perplexity", "bits/char"]])

plt.figure(figsize=(9, 4.5))
vals = abl_df["val_loss"]
colors = ["#1E2761" if k == "full" else "#F9A826" for k in vals.index]
plt.bar(vals.index, vals.values, color=colors)
lo, hi = vals.min(), vals.max()
plt.ylim(max(0, lo - 0.15), hi + 0.1)
plt.ylabel("validation loss (lower is better)"); plt.title(f"Ablation study ({N_ABL} steps each)")
for i, v in enumerate(vals.values):
    plt.text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)
plt.xticks(rotation=20); plt.grid(axis="y", alpha=0.25); plt.show()

### Interpreting the ablation (fill in from *your* results)

* **no_pos_emb** — the honest, interesting finding: a *causal* model can partly infer position because token *i* attends to exactly *i* earlier tokens (the amount of context differs by position), so removing positional embeddings often hurts **less than people expect** in decoder-only models. Report what you measured, and explain why. (Bidirectional encoders like BERT, which have no causal mask, would be hurt far more.)
* **heads** — with a fixed embedding size, changing the head count only changes *how* capacity is partitioned, not how much there is.
* **depth** — usually the largest effect: each layer lets the model compose the previous layer's patterns (characters → word pieces → words → phrases).

## 6. Decoding study — temperature, top-k, top-p

The trained model outputs a probability distribution over the next character; **decoding is how we pick from it**. We use the scaled model and measure two simple automatic metrics over 4 samples × 300 characters per setting:

* **valid-word rate** — fraction of generated words that actually occur in the training corpus (a proxy for *coherence / correctness*);
* **distinct-4-gram ratio** — unique 4-character chunks ÷ total (a proxy for *diversity*; greedy decoding loops, so it scores low).

Expected trade-off: low temperature → correct but repetitive; high temperature → varied but misspelled. Top-k / top-p try to get the best of both by cutting off the unlikely tail.

In [ ]:
def valid_word_rate(s):
    words = re.findall(r"[A-Za-z']+", s)
    return sum(w.lower() in corpus_words for w in words) / len(words) if words else 0.0

def distinct_ngrams(s, n=4):
    grams = [s[i:i + n] for i in range(len(s) - n + 1)]
    return len(set(grams)) / max(1, len(grams))

best = results["scaled"]["model"]
settings = [
    ("greedy (T=0)",        dict(temperature=0)),
    ("T=0.5",               dict(temperature=0.5)),
    ("T=0.8",               dict(temperature=0.8)),
    ("T=1.0",               dict(temperature=1.0)),
    ("T=1.4",               dict(temperature=1.4)),
    ("top-k=5, T=1.0",      dict(temperature=1.0, top_k=5)),
    ("top-k=40, T=1.0",     dict(temperature=1.0, top_k=40)),
    ("top-p=0.9, T=1.0",    dict(temperature=1.0, top_p=0.9)),
    ("top-p=0.9, T=0.8",    dict(temperature=0.8, top_p=0.9)),
]
ctx0 = torch.tensor([encode("ROMEO:")], dtype=torch.long, device=device)
rows, samples = [], {}
for name, kw in settings:
    torch.manual_seed(7)
    out = best.generate(ctx0.repeat(4, 1), 300, **kw)
    strs = [decode(o.tolist()) for o in out]
    samples[name] = strs[0]
    rows.append({"setting": name,
                 "valid_word_rate": round(float(np.mean([valid_word_rate(s) for s in strs])), 3),
                 "distinct_4gram": round(float(np.mean([distinct_ngrams(s) for s in strs])), 3)})
dec_df = pd.DataFrame(rows).set_index("setting")
display(dec_df)

for name in ["greedy (T=0)", "T=0.8", "T=1.4", "top-p=0.9, T=0.8"]:
    print(f"\n======== {name} ========\n{samples[name][:300]}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(dec_df["distinct_4gram"], dec_df["valid_word_rate"], s=90, color="#1E2761")
for name, r in dec_df.iterrows():
    ax.annotate(name, (r["distinct_4gram"], r["valid_word_rate"]), textcoords="offset points", xytext=(6, 5), fontsize=8)
ax.set_xlabel("diversity  (distinct 4-gram ratio)  →"); ax.set_ylabel("coherence  (valid-word rate)  →")
ax.set_title("Decoding trade-off: coherence vs diversity"); ax.grid(alpha=0.25); plt.show()

**Recommendation to state in the report:** use low-to-moderate temperature (≈0.6–0.8) with top-p ≈ 0.9 for factual / coherent output; raise temperature for creative output; avoid greedy for long text because it falls into repetition loops. (Check that your own plot supports this.)

## 7. Attention analysis — what do the heads actually look at?

Instead of only *describing* attention, we extract the **real attention matrices** from the trained scaled model.

* The strict lower-triangular shape is the **causal mask** in action — no head ever looks at the future.
* We also compute each head's **mean attention distance** (how far back it looks, in characters). Small = a *local* head (spelling, neighbouring letters); large = a *global* head (line/speaker structure).

In [ ]:
def show_attention(model, prompt, layer_ids):
    model.eval()
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for b in model.blocks: b.attn.store_attn = True
    with torch.no_grad():
        model(ids)
    for b in model.blocks: b.attn.store_attn = False
    labels = [{"\n": "↵", " ": "·"}.get(c, c) for c in prompt]
    H = model.cfg.n_head
    cols = min(4, H); rows_ = math.ceil(H / cols)
    for l in layer_ids:
        att = model.blocks[l].attn.last_attn[0]               # (H, T, T)
        fig, axes = plt.subplots(rows_, cols, figsize=(4.2 * cols, 4.2 * rows_))
        axes = np.array(axes).reshape(-1)
        for h in range(H):
            ax = axes[h]; ax.imshow(att[h].numpy(), cmap="viridis")
            ax.set_title(f"layer {l} · head {h}", fontsize=10)
            ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=6)
            ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=6)
            ax.set_xlabel("attended-to (key)", fontsize=7); ax.set_ylabel("query position", fontsize=7)
        for ax in axes[H:]: ax.axis("off")
        plt.suptitle(f"Attention maps — layer {l}", y=1.0); plt.tight_layout(); plt.show()

prompt = "First Citizen:\nBefore we proceed any"
show_attention(best, prompt, layer_ids=[0, best.cfg.n_layer - 1])

In [ ]:
def mean_attention_distance(model, passage):
    model.eval()
    ids = torch.tensor([encode(passage)], dtype=torch.long, device=device)
    for b in model.blocks: b.attn.store_attn = True
    with torch.no_grad():
        model(ids)
    for b in model.blocks: b.attn.store_attn = False
    T = ids.shape[1]
    pos = torch.arange(T)
    dist = (pos[:, None] - pos[None, :]).clamp(min=0).float()          # i - j for j <= i
    out = np.zeros((model.cfg.n_layer, model.cfg.n_head))
    for l, b in enumerate(model.blocks):
        att = b.attn.last_attn[0]                                      # (H, T, T)
        out[l] = (att * dist).sum(-1).mean(-1).numpy()
    return out

passage = val_text[:best.cfg.block_size]
dist_map = mean_attention_distance(best, passage)

plt.figure(figsize=(7, 4.5))
plt.imshow(dist_map, cmap="magma", aspect="auto")
plt.colorbar(label="mean attention distance (characters)")
plt.xlabel("head"); plt.ylabel("layer"); plt.title("Local vs global heads")
for (l, h), v in np.ndenumerate(dist_map):
    plt.text(h, l, f"{v:.0f}", ha="center", va="center", color="white", fontsize=8)
plt.show()
print("Smallest-distance head (most local):  layer %d head %d = %.1f chars" %
      (*np.unravel_index(dist_map.argmin(), dist_map.shape), dist_map.min()))
print("Largest-distance head  (most global): layer %d head %d = %.1f chars" %
      (*np.unravel_index(dist_map.argmax(), dist_map.shape), dist_map.max()))

## 8. Tokenisation study — character-level vs. BPE

Real LLMs use **sub-word (BPE) tokens**, not characters. Here we train a small byte-level BPE tokenizer (vocab 512) on the *training text only*, then train the **same architecture** on BPE tokens.

**Fairness caveat:** loss per token is *not* comparable across tokenisers (a BPE token carries more information than a character). The fair metric is **bits per character**:
`bits/char = (loss per token × tokens ÷ characters) ÷ ln 2`. This section needs `pip install tokenizers` (internet on Colab).

In [ ]:
try:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers"], check=True)
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

BPE_VOCAB = 512
tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=BPE_VOCAB, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
tok.train_from_iterator([train_text], trainer)

bpe_train = torch.tensor(tok.encode(train_text).ids, dtype=torch.long)
bpe_val = torch.tensor(tok.encode(val_text).ids, dtype=torch.long)
bpe_vocab = tok.get_vocab_size()
print("BPE vocab:", bpe_vocab,
      "| chars per token (train): %.2f" % (len(train_text) / len(bpe_train)),
      "| train tokens:", f"{len(bpe_train):,}")
demo = "To be, or not to be"
print("\nExample:", demo, "->", tok.encode(demo).tokens)

In [ ]:
torch.manual_seed(SEED)
N_BPE = ITERS(3000)
res_bpe = train_model("bpe_model", make_cfg(n_layer=4, n_head=4, n_embd=128, block_size=128, dropout=0.1,
                                            vocab_size=bpe_vocab),
                      max_iters=N_BPE, eval_interval=max(1, N_BPE // 10), data=(bpe_train, bpe_val))

tokens_per_char = len(bpe_val) / len(val_text)
res_bpe["bpc"] = res_bpe["val_loss"] * tokens_per_char / math.log(2)   # fair, per-character metric
results["bpe_model"] = res_bpe

ctxb = torch.tensor([tok.encode("ROMEO:").ids], dtype=torch.long, device=device)
print("\nBPE model sample:\n", tok.decode(res_bpe["model"].generate(ctxb, 120, temperature=0.8)[0].tolist()))

cmp_df = pd.DataFrame([
    {"tokeniser": "character (vocab 65)",   "val_bits_per_char": round(results["baseline"]["bpc"], 3),
     "params": results["baseline"]["params"]},
    {"tokeniser": f"BPE (vocab {bpe_vocab})", "val_bits_per_char": round(res_bpe["bpc"], 3),
     "params": res_bpe["params"]},
]).set_index("tokeniser")
display(cmp_df)

**Discussion points:** BPE shortens sequences (each token covers ~2–3 characters), so the same 128-token window sees *more text*; but the embedding/output tables are bigger and rarer tokens are seen less often. Compare the bits-per-character honestly — with a small model and corpus, the winner is not guaranteed, and reporting that candidly is better than assuming BPE must win.

## 9. Consolidated results

All runs in one table + a *parameters vs. validation loss* plot — a mini "scaling law" view.

In [ ]:
all_rows = [summarize(results[k]) for k in ("baseline", "scaled")]
all_rows += [summarize(r) for r in ablation.values()]
all_rows.append({**summarize(results["bpe_model"]), "bits/char": round(results["bpe_model"]["bpc"], 3)})
final_df = pd.DataFrame(all_rows).set_index("run")
display(final_df)
final_df.to_csv("all_results.csv")
print("saved -> all_results.csv")

plt.figure(figsize=(8, 5))
plt.scatter(final_df["params"], final_df["val_loss"], s=80, color="#1E2761")
for run, r in final_df.iterrows():
    plt.annotate(run.replace("abl_", ""), (r["params"], r["val_loss"]), textcoords="offset points", xytext=(5, 5), fontsize=8)
plt.xscale("log"); plt.xlabel("parameters (log scale)"); plt.ylabel("validation loss")
plt.title("Parameters vs validation loss (note: BPE loss is per-token, not comparable)")
plt.grid(alpha=0.25); plt.show()

## 10. How much did we improve, and why?

Jury 1 numbers below are the ones recorded in the first notebook (824,897 params, train 1.5640, val 1.7417, 2.22 min).
* ** 1 -> baseline re-run** isolates the effect of the *better training loop* (same architecture).
* **baseline re-run -> scaled** isolates the effect of *scale* (more layers, width, context, dropout).

In [ ]:
J1 = {"label": "Jury 1 model\n(recorded)", "params": 824_897, "train": 1.5640, "val": 1.7417, "minutes": 2.22}
rows = [J1,
        {"label": "Same architecture,\nimproved loop", "params": results["baseline"]["params"],
         "train": results["baseline"]["train_loss"], "val": results["baseline"]["val_loss"], "minutes": results["baseline"]["minutes"]},
        {"label": "Scaled model", "params": results["scaled"]["params"],
         "train": results["scaled"]["train_loss"], "val": results["scaled"]["val_loss"], "minutes": results["scaled"]["minutes"]}]
cmp = pd.DataFrame(rows)
cmp["perplexity"] = cmp["val"].apply(math.exp).round(2)
cmp["bits/char"] = (cmp["val"] / math.log(2)).round(3)
cmp["gap(val-train)"] = (cmp["val"] - cmp["train"]).round(4)
cmp["val_improvement_vs_J1"] = (cmp.loc[0, "val"] - cmp["val"]).round(4)
cmp["label"] = cmp["label"].str.replace("\n", " ")
display(cmp.set_index("label"))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
lbls = [l.replace(" ", "\n", 1) for l in cmp["label"]]
for ax, col, title in zip(axes, ["val", "perplexity", "params"], ["Validation loss (lower = better)", "Perplexity (lower = better)", "Parameters"]):
    ax.bar(lbls, cmp[col], color=["#9AA6D6", "#F9A826", "#1E2761"])
    ax.set_title(title, fontsize=11)
    for i, v in enumerate(cmp[col]):
        ax.text(i, v, f"{v:,.3f}" if col != "params" else f"{int(v):,}", ha="center", va="bottom", fontsize=8)
    ax.tick_params(axis="x", labelsize=8)
plt.tight_layout(); plt.show()

j1v, midv, lastv = cmp.loc[0, "val"], cmp.loc[1, "val"], cmp.loc[2, "val"]
print(f"Validation loss: {j1v:.4f} -> {midv:.4f} -> {lastv:.4f}")
print(f"  from better training loop alone : {j1v - midv:+.4f} ({(j1v - midv) / j1v * 100:.1f}% lower)")
print(f"  additional from scaling         : {midv - lastv:+.4f} ({(midv - lastv) / midv * 100:.1f}% lower)")
print(f"  total vs Jury 1                 : {j1v - lastv:+.4f} ({(j1v - lastv) / j1v * 100:.1f}% lower) | "
      f"perplexity {math.exp(j1v):.2f} -> {math.exp(lastv):.2f} | params x{cmp.loc[2, 'params'] / J1['params']:.1f}")

## 11. Save, reload, and interactive generation

The best scaled checkpoint was saved during training. Below we prove it reloads and generates identically — useful for a live demo without retraining.

In [ ]:
def load_model(path):
    ck = torch.load(path, map_location=device)
    m = MiniGPT(Config(**ck["cfg"])).to(device)
    m.load_state_dict(ck["state"]); m.eval()
    return m

def complete(model, prompt, n=300, **kw):
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    return decode(model.generate(ids, n, **kw)[0].tolist())

reloaded = load_model("scaled_best.pt")
print(complete(reloaded, "ROMEO:", n=400, temperature=0.8, top_p=0.9))

In [ ]:
# Live demo cell: change the prompt / settings and re-run
print(complete(reloaded, "KING HENRY:\nWhat news", n=400, temperature=0.7, top_k=40))